In [2]:
import pandas as pd
import numpy as np

## Background and Motivation

Columns that contain many repeated values from a small set of distinct options are common in real datasets.

A naive string representation stores each value as a full Python object, which is memory-heavy. A better approach — used in data warehousing — is the **dictionary-encoded** (categorical) representation:

- Store a **dimension table** of the distinct values (the *categories*).
- Store the actual data as **integer codes** pointing into the dimension table.
- Reconstruct the original values with `dim.take(codes)`.

### Terminology

| Term | Meaning |
|------|---------|
| Categories / levels | The distinct unique values |
| Codes | Integer indices referencing the categories |
| Categorical / dictionary-encoded | The representation as (codes, categories) |

### Benefits

- Significant memory savings for columns with low cardinality (few unique values).
- Faster GroupBy and aggregation operations — algorithms operate on integer codes.
- Low-cost category-level transforms: rename, reorder, or append categories without touching the codes.

## Key Points

- Dictionary encoding stores data as integer codes + a small lookup table of unique values.
- `dim.take(codes)` reconstructs the original string Series from codes and a dimension table.
- The main gains are memory (int8 codes vs full Python strings) and computation speed.

In [3]:
values = pd.Series(['apple', 'orange', 'apple', 'apple'] * 2)

print(pd.unique(values))
print()
print(values.value_counts())

['apple' 'orange']

apple     6
orange    2
Name: count, dtype: int64


In [4]:
# Dictionary-encoded representation: integer codes + dimension table
codes = pd.Series([0, 1, 0, 0] * 2)
dim = pd.Series(['apple', 'orange'])

dim.take(codes)  # reconstruct original Series

0     apple
1    orange
0     apple
0     apple
0     apple
1    orange
0     apple
0     apple
dtype: object

## Categorical Extension Type in pandas

pandas has a `Categorical` extension type that formalizes the dictionary-encoded representation.

### Creating a Categorical

| Method | Use case |
|--------|---------|
| `series.astype("category")` | Convert an existing Series |
| `pd.Categorical(list)` | Create directly from a Python sequence |
| `pd.Categorical.from_codes(codes, categories)` | Create from pre-encoded integer codes |

### Inspecting a Categorical

- `.array` — the underlying `Categorical` object.
- `.categories` — `Index` of unique values.
- `.codes` — `int8` array of integer codes.
- `dict(enumerate(c.categories))` — quick code → category lookup map.

### Ordering

By default categoricals are **unordered** — no comparison between categories is implied.

- Pass `ordered=True` to `from_codes` (or other constructors) to declare a meaningful order.
- Output shows `'foo' < 'bar' < 'baz'` to indicate ordering.
- Call `.as_ordered()` on an existing unordered Categorical to add ordering.

## Key Points

- `astype("category")` is the most common conversion path.
- The underlying `Categorical` object is accessed via `.array`.
- `.codes` is `int8` — very compact compared to Python string objects.
- `ordered=True` enables meaningful comparisons and sorting between categories.
- Categories can be any immutable type — not just strings.

In [5]:
rng = np.random.default_rng(seed=12345)
fruits = ['apple', 'orange', 'apple', 'apple'] * 2
N = len(fruits)

df = pd.DataFrame({
    'basket_id': np.arange(N),
    'fruit': fruits,
    'count': rng.integers(3, 15, size=N),
    'weight': rng.uniform(0, 4, size=N)
}, columns=['basket_id', 'fruit', 'count', 'weight'])

df

,basket_id,fruit,count,weight
0,0,apple,11,1.564438
1,1,orange,5,1.331256
2,2,apple,12,2.393235
3,3,apple,6,0.746937
4,4,apple,5,2.691024
5,5,orange,12,3.767211
6,6,apple,10,0.992983
7,7,apple,11,3.795525


In [6]:
fruit_cat = df['fruit'].astype('category')

fruit_cat

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, object): ['apple', 'orange']

In [7]:
c = fruit_cat.array

print(type(c))
print(c.categories)
print(c.codes)

<class 'pandas.core.arrays.categorical.Categorical'>
Index(['apple', 'orange'], dtype='object')
[0 1 0 0 0 1 0 0]


In [8]:
dict(enumerate(c.categories))  # code → category lookup

{0: 'apple', 1: 'orange'}

In [9]:
# Assign back to convert the column in the DataFrame
df['fruit'] = df['fruit'].astype('category')

df['fruit']

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, object): ['apple', 'orange']

In [10]:
# Create directly from a Python list
pd.Categorical(['foo', 'bar', 'baz', 'foo', 'bar'])

['foo', 'bar', 'baz', 'foo', 'bar']
Categories (3, object): ['bar', 'baz', 'foo']

In [11]:
# Create from pre-existing codes and categories
categories = ['foo', 'bar', 'baz']
codes = [0, 1, 2, 0, 0, 1]

my_cats = pd.Categorical.from_codes(codes, categories)

my_cats

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo', 'bar', 'baz']

In [12]:
# ordered=True: declares a meaningful ordering between categories
ordered_cat = pd.Categorical.from_codes(codes, categories, ordered=True)

ordered_cat

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

In [13]:
my_cats.as_ordered()  # add ordering to an unordered categorical

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

## Computations with Categoricals

Categorical data integrates directly into pandas operations and often yields better performance.

### With `qcut`

`pd.qcut` returns a Categorical by default. Use `labels=` to name the quartiles, then use `groupby` to compute summary statistics per bin — the groupby key retains category ordering.

### Performance

For large Series with low cardinality:

- **Memory**: `int8` codes + small category table vs. full Python string objects.
- **Speed**: GroupBy and aggregation operate on integer codes — significantly faster than string-based operations.

The conversion to `category` is a one-time cost, after which repeated operations are much cheaper.

## Key Points

- `pd.qcut` with `labels=` returns a labeled ordered Categorical, ready for groupby.
- Categorical columns in groupby keys retain ordering in the result.
- For 10M rows with 4 categories: ~60× memory reduction, ~28× faster `value_counts`.
- Conversion cost is paid once; savings compound across repeated operations.

In [14]:
rng = np.random.default_rng(seed=12345)
draws = rng.standard_normal(1000)

bins = pd.qcut(draws, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

print(bins[:5])
print()
print(bins.codes[:10])

['Q1', 'Q4', 'Q1', 'Q2', 'Q2']
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

[0 3 0 1 1 0 0 2 2 0]


In [15]:
# groupby on a categorical key — ordering is preserved in results
bins_s = pd.Series(bins, name='quartile')

results = (
    pd.Series(draws)
    .groupby(bins_s)
    .agg(['count', 'min', 'max'])
    .reset_index()
)

results

/tmp/ipykernel_7641/1765414643.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(bins_s)


,quartile,count,min,max
0,Q1,250,-3.119609,-0.678494
1,Q2,250,-0.673305,0.008009
2,Q3,250,0.018753,0.686183
3,Q4,250,0.688282,3.211418


In [16]:
results['quartile']  # retains categorical dtype and ordering

0    Q1
1    Q2
2    Q3
3    Q4
Name: quartile, dtype: category
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

In [17]:
# Memory comparison: string Series vs categorical Series
N = 10_000_000
labels = pd.Series(['foo', 'bar', 'baz', 'qux'] * (N // 4))
categories = labels.astype('category')

print(f"string dtype:      {labels.memory_usage(deep=True):>12,} bytes")
print(f"category dtype:    {categories.memory_usage(deep=True):>12,} bytes")

string dtype:       600,000,128 bytes
category dtype:      10,000,540 bytes


In [18]:
# Speed comparison: value_counts on string vs categorical
%timeit labels.value_counts()

433 ms ± 7.31 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%timeit categories.value_counts()

40.8 ms ± 2.98 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Categorical Methods — `cat` Accessor

Series with `category` dtype expose a `cat` accessor — similar to `str` for strings.

### Key Attributes

| Accessor | Returns |
|----------|---------|
| `cat.codes` | Integer code Series |
| `cat.categories` | Index of category labels |

### Key Methods

| Method | Effect |
|--------|-------|
| `set_categories(new)` | Replace the full category list (can add or remove) |
| `add_categories(new)` | Append new unused categories |
| `remove_categories(old)` | Remove categories, setting affected values to null |
| `remove_unused_categories()` | Drop categories that have no observations |
| `rename_categories(new)` | Rename categories in place (count must stay the same) |
| `reorder_categories(new)` | Like rename but can also set ordering |
| `as_ordered()` | Make categories ordered |
| `as_unordered()` | Make categories unordered |

### Why `set_categories` Matters

After filtering a large dataset, the original categories list may still contain values that no longer appear in the data. `value_counts` will show zeros for those categories. Call `remove_unused_categories` to clean up.

## Key Points

- `cat.codes` and `cat.categories` are the low-level building blocks of a Categorical.
- `set_categories` changes the reference list without modifying the data.
- `value_counts` on a Categorical with extra categories shows zero-count rows for unobserved values.
- After filtering, call `remove_unused_categories` to trim phantom categories.

In [20]:
s = pd.Series(['a', 'b', 'c', 'd'] * 2)
cat_s = s.astype('category')

print(cat_s.cat.codes)
print()
print(cat_s.cat.categories)

0    0
1    1
2    2
3    3
4    0
5    1
6    2
7    3
dtype: int8

Index(['a', 'b', 'c', 'd'], dtype='object')


In [21]:
# set_categories: expand the known category list to include 'e'
cat_s2 = cat_s.cat.set_categories(['a', 'b', 'c', 'd', 'e'])

cat_s2

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (5, object): ['a', 'b', 'c', 'd', 'e']

In [22]:
# value_counts respects the full category list — shows 0 for unobserved 'e'
print("Without 'e' in categories:")
print(cat_s.value_counts())
print()
print("With 'e' in categories:")
print(cat_s2.value_counts())

Without 'e' in categories:
a    2
b    2
c    2
d    2
Name: count, dtype: int64

With 'e' in categories:
a    2
b    2
c    2
d    2
e    0
Name: count, dtype: int64


In [23]:
# After filtering, unused categories remain in the list
cat_s3 = cat_s[cat_s.isin(['a', 'b'])]

print(cat_s3)
print()

0    a
1    b
4    a
5    b
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']



In [24]:
# remove_unused_categories trims categories with no observations
cat_s3.cat.remove_unused_categories()

0    a
1    b
4    a
5    b
dtype: category
Categories (2, object): ['a', 'b']

## Creating Dummy Variables for Modeling

Statistical models and machine learning algorithms typically require numerical input. Categorical variables are converted to **dummy variables** (one-hot encoding) via `pd.get_dummies`:

- One binary column per category.
- `1` where the value matches the category, `0` elsewhere.

When the input is already a `category` dtype, `get_dummies` respects the full category list — including any categories with zero observations.

## Key Points

- `pd.get_dummies(categorical_series)` → one binary column per category.
- Works on `category` dtype Series directly — no need to convert back to strings first.
- Combines naturally with `pd.cut` / `pd.qcut` for binned numeric features.

In [25]:
cat_s = pd.Series(['a', 'b', 'c', 'd'] * 2, dtype='category')

pd.get_dummies(cat_s)

,a,b,c,d
0,True,False,False,False
1,False,True,False,False
2,False,False,True,False
3,False,False,False,True
4,True,False,False,False
5,False,True,False,False
6,False,False,True,False
7,False,False,False,True
